# Oracle4SD Atlas: from native evidence to economic measurements

This tutorial follows the Template's eight stages. It reconstructs one UMA settlement, checks bounded native samples for eight oracle branches, and inspects the daily measurement product. It does not collect every historical record. Each section states its observation unit and output.

| Section | Output | Main code |
|---|---|---|
| [1. Setup](#1-setup) | Version and environment record | `scripts/bootstrap_environment.py` |
| [2. Sources](#2-sources-and-metadata) | Source, scope and rights tables | `metadata/acquisition_profiles.json` |
| [3. Query](#3-querying) | Raw responses and acquisition receipts | `code/query_data/query.py`, `tutorial_revision/query.py` |
| [4. Inspect raw data](#4-queried-data-inspection) | Receipt and event locators | case source specifications |
| [5. Process](#5-processing) | One linked episode and eight sample checks | `code/process_data/process.py` |
| [6. Inspect tables](#6-processed-data-inspection) | Dictionaries, keys and physical inventory | `metadata/revision/table_catalog.json` |
| [7. Describe](#7-descriptive-analysis) | Accounting, missingness and daily summaries | `code/analyze_data/analyze.py` |
| [8. Validate and export](#8-validation-and-reproduction) | Check matrix and evidence ZIP | `code/technical_validation/validate.py` |

An oracle provides claims to a blockchain program. A receipt records transaction execution; its logs record contract events. A reward is distinct from a return of the participant's principal. Protocol resolution does not independently verify an external fact.

## 1. Setup

Use a CPU runtime. The case uses a managed Python 3.12 environment; no key or wallet is needed for archived replay. The small native samples are bundled. Full-table downloads are separate. See the English acceptance guide in `docs/COLAB_ACCEPTANCE.md` for restart, export and independent-reader instructions.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, platform, shutil, subprocess, sys, tempfile, time
from IPython.display import display, Markdown, FileLink
CODE_REF = "71c0e19d3ae2b47d763243371f1bfb97750db61d"
WORK = Path(tempfile.mkdtemp(prefix="oracle-atlas-"))
def command(args, cwd=None):
    env=dict(os.environ, MPLBACKEND="Agg")
    result=subprocess.run([str(a) for a in args],cwd=cwd,env=env,capture_output=True,text=True)
    with (WORK/"commands.log").open("a") as f:
        f.write(json.dumps([str(a) for a in args])+"\n"+result.stdout+result.stderr+"\n")
    if result.returncode:raise RuntimeError(result.stdout+result.stderr)
    return result.stdout
search=[Path.cwd(),Path.cwd().parent]
REPO=next((p for p in search if (p/"configs/atlas_utc_release.json").exists()),None)
if REPO is None:
    REPO=WORK/"repository"
    command(["git","clone","--quiet","https://github.com/Oracle4CEG/Oracle4SD-Atlas.git",REPO])
    command(["git","checkout","--detach",CODE_REF],REPO)
PYTHON=os.environ.get("ORACLE_NOTEBOOK_PYTHON")
if not PYTHON:
    command([sys.executable,REPO/"scripts/bootstrap_environment.py","--output",WORK/"environment"])
    PYTHON=str(WORK/"environment/scientific/bin/python")
import pandas as pd
def read_json(p):return json.loads(Path(p).read_text())
RUN=WORK/"case";RUN.mkdir()
NATIVE=WORK/"native"
shutil.copytree(REPO/"tutorial_revision",NATIVE,ignore=shutil.ignore_patterns("*.ipynb","__pycache__"))
sys.path.insert(0,str(NATIVE));sys.path.insert(0,str(NATIVE/"processing_code"))
from query import query
from replay import run_branch
profiles=read_json(NATIVE/"metadata/profiles.json")
raw_profiles=read_json(NATIVE/"metadata/raw_profiles.json")
def stage(folder, script, extra=(), run=RUN):
    return command([PYTHON,REPO/"code"/folder/script,"--config",REPO/"configs/project.json","--run-dir",run,*extra],REPO)
identity={"code_commit":command(["git","rev-parse","HEAD"],REPO).strip(),
          "working_tree_modified":bool(command(["git","status","--porcelain"],REPO).strip()),
          "data_release":read_json(REPO/"configs/atlas_utc_release.json"),
          "runtime_kind":"hosted_colab" if "google.colab" in sys.modules else "local_jupyter_kernel",
          "host_python":platform.python_version(),"independent_reader":False}
display(identity)

## 2. Sources and metadata

The first register describes seven case files covering five transactions. The historical full-source register has 64 native tables and six archive metadata entries. Native tables are retained collector outputs, not a complete archive of original RPC responses. The current inventory separately describes 100 physical tables.

The eight branches have different scopes. UMA, Chainlink, Tellor, Flare FTSOv2 and Pyth are detailed panels; Chronicle, RedStone Push and DIA are narrower extensions. The acquisition profiles state chains, implementations, range endpoints, source methods and limitations. Project evidence is distributed under CC BY 4.0; upstream rights are retained in `DATA_LICENSE.md`.

In [ ]:
display(pd.read_csv(REPO/"metadata/source_register.csv"))
full_sources=pd.read_csv(REPO/"metadata/full_source_register_20261012.csv")
print("Historical source register:",len(full_sources),"entries")
display(full_sources.head(4))
scope=pd.DataFrame(read_json(REPO/"metadata/acquisition_profiles.json")["profiles"])
display(scope[[c for c in ["protocol","chain","delivery_chain","contracts","start","end","limitations"] if c in scope]])
display(pd.DataFrame(profiles)[["branch","parent_native_table","source_mode","sample_rows","acquisition_scope"]])

## 3. Querying

| Mode | What is obtained | Boundary |
|---|---|---|
| Bundled case | Seven registered files | No fresh API requests |
| Public download | Two pinned receipt snapshots; five bundled support files | Secondary archive download |
| Live case | Five receipts, chain identity, headers and retained code | Fixed case; may encounter access failures |
| Eight-branch replay | One registered raw response and a native subset per branch | Bounded identity checks |

Archived replay is the default. Set `RUN_NETWORK_EXAMPLES=True` to run the separate public-download and live cases. Live responses must pass locator and chain checks; they need not have the same response-envelope bytes as an archive. The eight-branch commands in the acquisition guide each stop after one registered transaction or height, use a 25-second timeout, and retain an error on failure.

In [ ]:
stage("query_data","query.py",["--mode","bundled"])
acquisition=read_json(RUN/"reports/acquisition.json")
assert acquisition["source_transactions"]==5
assert all(f["frozen_hash_matches"] for f in acquisition["files"])
raw_checks=[query(NATIVE,p["branch"],"archived_raw") for p in raw_profiles]
display(pd.DataFrame(raw_checks)[["branch","mode","expected_native_block","locator_verified"]])
RUN_NETWORK_EXAMPLES=False
if RUN_NETWORK_EXAMPLES:
    for mode in ["public-download","live"]:
        destination=WORK/mode;destination.mkdir()
        try:
            stage("query_data","query.py",["--mode",mode],destination)
            report=read_json(destination/"reports/acquisition.json")
            if mode=="public-download":assert all(f["frozen_hash_matches"] for f in report["files"])
            stage("process_data","process.py",run=destination)
            stage("technical_validation","validate.py",run=destination)
            display(report)
        except RuntimeError as error:
            (destination/"failure.txt").write_text(str(error))
            print(mode,"failed; retained diagnostic, no empty replacement")
else:
    print("Network examples not executed in this run. Retained bounded query evidence:")
    display(pd.DataFrame(read_json(REPO/"reports/publication_20261029/bounded_acquisition.json"))[["branch","mode","status","elapsed_seconds"]])

## 4. Queried-data inspection

`transactionHash` locates the transaction; `blockHash` identifies the block; `logIndex` locates an emitted event within its chain log order. The raw integer block number is hexadecimal in an Ethereum JSON-RPC receipt. A decoded event is shown after processing, with the source transaction retained. Neither receipt success nor a matching locator proves the external claim was true.

In [ ]:
snapshot=read_json(RUN/"queried_data/uma_case/raw/polygon_rpc_snapshot.json")
receipt=next(iter(snapshot["receipts"].values()))
preview={k:receipt.get(k) for k in ["transactionHash","blockHash","blockNumber","status"]}
preview["first_log"]={k:receipt["logs"][0].get(k) for k in ["address","logIndex","topics","data"]}
display(preview)
display(Markdown("[Original transaction](https://polygonscan.com/tx/"+receipt["transactionHash"]+")"))

## 5. Processing

| Transformation | Input → output unit | Key or rule |
|---|---|---|
| ABI decoding | Receipt logs → 9 selected events | Chain, transaction and log index |
| Token-flow selection | Transfer logs → 8 USDC.e movements | Token contract and source locator |
| Lifecycle linkage | 5 transactions → 1 episode | OOv2 request and DVM identifier |
| Settlement accounting | Gross payment → principal and gain | Integer token units; principal is separated |
| Native sample replay | Bounded table → clock and identity check | Registered SHA-256 and row count |

The eight sample checks verify retained native inputs; they do not derive the full native history from the single raw response. That boundary is deliberate and appears in every sample profile.

In [ ]:
stage("process_data","process.py")
processing=read_json(RUN/"reports/processing.json")
display(processing["counts"])
case_events=pd.read_csv(RUN/"processed_data/source_events.csv")
display(case_events.head(1))
sample_checks=[]
for p in profiles:
    checked=run_branch(NATIVE,p["branch"]);sample_checks.append(checked)
    sample=pd.read_parquet(NATIVE/p["input"])
    cols=[c for c in ["event_time_utc","epoch_start_utc","epoch_end_utc","event_time_status"] if c in sample]
    print(p["branch"],"—",p["parent_native_table"],"—",p["sample_rows"],"retained rows")
    display(sample[cols].head(2))
display(pd.DataFrame(sample_checks))

## 6. Processed-data inspection

The episode dictionary defines all 85 fields. Case child tables describe events, transfers, accounting components and checks for the same episode; their row counts are not additional lifecycles. The single-case join is the retained request/episode identity. Source-event and transfer identities also include chain, transaction and log position.

The current full catalogue contains 100 physical tables, replacing the previous 95-table inventory for this release. Shared layers overlap. Clock companions join their parent records by record identifier; filter the parent branch before aggregation. `event_time_utc` is an observed execution time, epoch bounds are interval support, and `event_time_status` states the basis or absence of an event clock. Missing times remain null. Dictionary definition status distinguishes complete revision rules from native fields requiring further semantic review.

In [ ]:
episode=read_json(RUN/"processed_data/economic_episode.json")
dictionary=pd.read_csv(REPO/"metadata/data_dictionary.csv")
assert set(dictionary.field_name)==set(episode)
display(dictionary[["field_name","type_or_enum","unit","missing_policy"]].head(8))
case_tables=read_json(REPO/"metadata/table_dictionary.json")["tables"]
display(pd.DataFrame.from_dict(case_tables,orient="index"))
print("The tenth case artifact is economic_episode.json: the same episode as economic_episodes.csv, not an additional observation.")
catalog=pd.DataFrame(read_json(REPO/"metadata/revision/table_catalog.json"))
assert len(catalog)==100
display(catalog[["path","products","layer","observation_unit","rows"]])
full_dictionary=pd.read_csv(REPO/"metadata/revision/field_dictionary.csv")
display(full_dictionary[full_dictionary.field.isin(["event_time_utc","epoch_start_utc","epoch_end_utc","event_time_status"])][["table_path","field","arrow_type","definition","definition_status"]].head(8))
croissant=read_json(REPO/"metadata/revision/croissant.json")
assert len(croissant["recordSet"])==100
print("Croissant recordsets:",len(croissant["recordSet"]),"; metadata parsing is separate from data validation.")

## 7. Descriptive analysis

The case payment is 1,000 USDC.e: 750 returned principal plus 250 token gain. The configured request reward is a different quantity. USDC.e has six decimals; gas uses another asset and is not silently deducted.

The daily table is a bounded full-release inspection: 13,269 typed rows, with source and timed-population counts. A row count describes released groups, not the number of independent participants. Missingness below uses one episode per field as its denominator; it is tutorial-only. The paper map names which broader products support manuscript figures and appendices.

In [ ]:
stage("analyze_data","analyze.py")
summary=read_json(RUN/"analysis/summary.json")
principal=int(summary["principal_returned_raw"]);gain=int(summary["realized_payoff_raw"]);gross=int(summary["gross_payout_raw"])
assert principal+gain==gross==1000000000
display(pd.DataFrame([{"component":"returned principal","raw_units":principal},{"component":"token gain","raw_units":gain},{"component":"gross payment","raw_units":gross}]))
missing=pd.DataFrame([{"field":k,"missing":int(v is None),"denominator":1,"unit":"episode"} for k,v in episode.items()])
missing.to_csv(RUN/"analysis/missingness_with_denominators.csv",index=False)
display(missing[missing.missing==1])
daily=pd.read_parquet(NATIVE/"data_source/derived/daily_measurements.parquet")
assert len(daily)==13269
display(daily.groupby("branch").agg(rows=("branch","size"),coverage=("coverage_status",lambda s:", ".join(sorted(set(s))))))
display(daily.groupby(["branch","mechanism_version","metric"])[["source_population_count","source_timed_count"]].max())
display(pd.read_csv(REPO/"metadata/revision/paper_data_code_map.csv"))

## 8. Validation and reproduction

Case checks, sample identity checks and full-release validation have different populations. A deterministic rerun compares scientific outputs from the same inputs and code. It does not substitute for another reader's attempt or Google-hosted execution. Source completeness, historical parameter bindings and independent factual truth are not established by these tests.

The export includes only the public case, native sample outputs and runtime records. It excludes credentials and the full dataset. Source rights remain those in `DATA_LICENSE.md`. The English acceptance guide explains how an independent reader should retain interventions and report discrepancies.

In [ ]:
stage("technical_validation","validate.py")
validation=read_json(RUN/"reports/validation.json")
assert validation["passed"] and validation["passed_checks"]==28
display(pd.DataFrame(validation["checks"]))
display(pd.read_csv(REPO/"metadata/completion_audit/validation_scope.csv"))
SECOND=WORK/"case-repeat"
command([PYTHON,REPO/"scripts/run_pipeline.py","--output",SECOND],REPO)
assert read_json(RUN/"analysis/summary.json")==read_json(SECOND/"analysis/summary.json")
EVIDENCE=WORK/"evidence";EVIDENCE.mkdir()
shutil.copytree(RUN,EVIDENCE/"case")
shutil.copytree(SECOND,EVIDENCE/"case-repeat")
shutil.copytree(NATIVE/"processed_data",EVIDENCE/"native-sample-results")
shutil.copy2(WORK/"commands.log",EVIDENCE/"commands.log")
(EVIDENCE/"identity.json").write_text(json.dumps(identity,indent=2))
(EVIDENCE/"scientific.freeze.txt").write_text(command([PYTHON,"-m","pip","freeze"]))
inventory=[{"path":str(p.relative_to(EVIDENCE)),"bytes":p.stat().st_size} for p in sorted(EVIDENCE.rglob("*")) if p.is_file()]
display(pd.DataFrame(inventory))
(EVIDENCE/"inventory.json").write_text(json.dumps(inventory,indent=2))
export=shutil.make_archive(str(WORK/"oracle-atlas-evidence"),"zip",EVIDENCE)
display(FileLink(export))
print("Case checks and deterministic repeat passed. Independent-reader acceptance remains separate.")

### Paper, data and acceptance

The current manuscript is `paper/working_revision/main.pdf`; Appendix B includes native-to-UTC examples. The dataset's ten products and fixed revision are recorded in `configs/atlas_utc_release.json`. Use `scripts/load_atlas_product.py CONFIG` to list all associated tables, or add `--output DIRECTORY` to retrieve the product's non-shared tables with hash checks. Shared tables require explicit `--include-shared` and subsequent branch/key filtering.

Cite Oracle4CEG, *Oracle4SD Atlas: eight oracle branches, UMA tasks and daily measurements*, version `atlas-utc-20261029`, Hugging Face (2026), together with the resolved dataset commit. No DOI is assigned. The legacy experimental configuration remains available at the predecessor revision recorded in release metadata.

For a failed checksum, use the pinned input again and preserve the failure. For an unavailable historical endpoint, retain the acquisition error and use archived replay only for its declared scope. Restart the kernel and run all cells after an interrupted execution. Report reproducibility issues through the Oracle4SD-Atlas GitHub repository, including the evidence ZIP and any intervention; do not include credentials.